# Named Entity Recognition with Google Gemini

In this notebook, we'll process a text for NER using the free API of Google Gemini. The API is stored locally in the "secrects" of this notebook. You can download your own API from Google AI Studio.

## Process Overview

The process consists of the following steps:
1. **Text Input**: We start with plain text that needs entity recognition
2.  **LLM Processing**: The text is sent to GPT with a prompt that instructs it to identify entities
3. **Entity Annotation**: The LLM marks entities in markdown format: `[Entity](TYPE)`
4. **Visualization**: The annotated text is displayed with color-coded entity highlighting

This approach leverages the LLM's natural language understanding while producing structured, machine-readable output.

## Install Packages

First, we will install the required dependencies:
- `google.generativeai`
- `python-dotenv` (only necessary if working on your local device)
- `spaCy` and `pandas` for visualization and analysis of the results

In [1]:
%pip install google-generativeai python-dotenv pydantic spacy pandas

### Load the Google Gemini API

**To work on Google Colab:**
1. Get your API key from [Google AI Studio](https://makersuite.google.com/app/apikey)
2. Import the API key in the secrets of this notebook
3. Import the necessary packages and the api key with
```python
from google.colab import userdata
userdata.get('YOUR_API_KEY_NAME')
```

**To work on your device:**
1. Get your API key from [Google AI Studio](https://makersuite.google.com/app/apikey)
2. Create a `.env` file in the same directory with: `GOOGLE_API_KEY=your_actual_api_key_here`
3. Replace `your_actual_api_key_here` with your real API key
4. Call the API key with:
```python
from dotenv import load_dotenv
load_dotenv()
GOOGLE_API_KEY = os.getenv('YOUR_API_KEY_NAME')
```

Calling the API differs slightly depending on whether you're working on your own devide or on this notebook.

The first time you do this, you may have to import your api key from Google AI Studio and grant access to this specific notebook.

In [2]:
# uncomment if working on own device, to allow access to parent directory.
#import sys
#sys.path.append("..")

In [3]:
import os
import re
import google.generativeai as genai
from pydantic import BaseModel
import json
import pandas as pd

import spacy
from spacy import displacy

# we import the necessary packages to call the API from this notebook's secrets.
from google.colab import userdata

In [4]:
# necessary packages to call the API from this notebook's secrets.
#from google.colab import userdata
#api_key = userdata.get('GOOGLE_API_KEY')

# uncomment if working on own device, to allow loading the .env variable.

#from dotenv import load_dotenv
#load_dotenv()
#GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

### Configure the Google Gemini Client

The first time you do this, you may have to import your api key from Google AI Studio and grant access to this specific notebook.

In [5]:
genai.configure(api_key=userdata.get('GOOGLE_API_KEY')) # if working on this colab notebook
#genai.configure(api_key=YOUR_API_KEY) # if working on own device
model = genai.GenerativeModel('gemini-2.5-flash') # here we define the model we want to work with

### List available models

Optionally, we may list all the models available with our API and change the specific model we want to use.

We may select any of these models and set it as our generative model when we configure the Gemini client. By default, the free API key runs with gemini 2.5-flash.  

```python
model = genai.GenerativeModel('your-preferred-model')
```

In [6]:
# List all the models available under the genai client.

import google.generativeai as genai
# call the api_key from the secrets in this colab notebook
genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))
for m in genai.list_models():
    # Check available methods, e.g., 'generateContent' support
    if "generateContent" in getattr(m, "supported_generation_methods", []):
        print(m.name)

models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash
models/gemini-2.5-flash-lite-preview-06-17
models/gemini-2.5-pro-preview-05-06
models/gemini-2.5-pro-preview-06-05
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-pro-exp
models/gemini-2.0-pro-exp-02-05
models/gemini-exp-1206
models/gemini-2.0-flash-thinking-exp-01-21
models/gemini-2.0-flash-thinking-exp
models/gemini-2.0-flash-thinking-exp-1219
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/learnlm-2.0-flash-experimental
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.

## Functions for this notebook

In order to visualize our annotated text with displaCy, we need to convert it into a format that displaCy can read.

The functions below adapt a similar one from our previous experiment with GLiNER, and converts the output of our NER process into a format that displaCy can understand.

The function `parse_annotated_text_to_dataframe()` takes the markdown output in the format `[Entity](LABEL)` and converts it into a pandas dataframe, so that we can save the results as a csv file.

In [7]:
from spacy.tokens import Doc, Span

def annotated_text_to_spacy_doc(text, nlp=None):
    """
    Converts annotated text in format [Entity](LABEL) to a spaCy Doc with entity spans.

    Args:
        text (str): Text with annotations like "[Tom](PERSON) worked for [Microsoft](ORGANIZATION)"
        nlp (spacy.Language, optional): spaCy language model. If None, uses blank English model.

    Returns:
        spacy.tokens.Doc: spaCy document with entity spans set

    Example:
        >>> text = "[Tom](PERSON) worked for [Microsoft](ORGANIZATION) in 2020 before he lived in [Rome](LOCATION)."
        >>> doc = annotated_text_to_spacy_doc(text)
        >>> spacy.displacy.render(doc, style="ent")
    """
    if nlp is None:
        nlp = spacy.blank("en")

    # Pattern to match [text](LABEL) format
    pattern = r'\[([^\]]+)\]\(([^)]+)\)'

    # Parse the text to extract tokens and entity information
    tokens = []
    entity_spans = []  # List of (start_token_idx, end_token_idx, label)
    custom_labels = set()

    # Split text by the pattern and process each part
    last_end = 0
    token_idx = 0

    for match in re.finditer(pattern, text):
        # Add tokens before the entity
        before_entity = text[last_end:match.start()]
        if before_entity.strip():
            # Tokenize the text before the entity
            before_tokens = before_entity.split()
            tokens.extend(before_tokens)
            token_idx += len(before_tokens)

        # Add the entity tokens
        entity_text = match.group(1)
        entity_label = match.group(2)
        custom_labels.add(entity_label)

        # Tokenize the entity text
        entity_tokens = entity_text.split()
        start_token_idx = token_idx
        tokens.extend(entity_tokens)
        token_idx += len(entity_tokens)
        end_token_idx = token_idx

        # Store entity span information
        entity_spans.append((start_token_idx, end_token_idx, entity_label))

        last_end = match.end()

    # Add any remaining tokens after the last entity
    remaining = text[last_end:]
    if remaining.strip():
        remaining_tokens = remaining.split()
        tokens.extend(remaining_tokens)

    # Add custom labels to the NLP model if they don't exist
    if "ner" not in nlp.pipe_names:
        ner = nlp.add_pipe("ner")
    else:
        ner = nlp.get_pipe("ner")

    for label in custom_labels:
        ner.add_label(label)

    # Create spaces array (True for tokens that should have a space after them)
    # Simple heuristic: all tokens except the last one get a space
    spaces = [True] * len(tokens)
    if tokens:
        spaces[-1] = False

    # Create the Doc from tokens
    doc = Doc(nlp.vocab, words=tokens, spaces=spaces)

    # Create entity spans
    entities = []
    for start_idx, end_idx, label in entity_spans:
        if start_idx < len(doc) and end_idx <= len(doc):
            span = Span(doc, start_idx, end_idx, label=label)
            entities.append(span)

    # Set entities on the document
    doc.ents = entities

    return doc


def visualize_annotated_text(text, nlp=None, style="ent", jupyter=True):
    """
    Convenience function to convert annotated text and visualize it with displaCy.

    Args:
        text (str): Text with annotations like "[Tom](PERSON) worked for [Microsoft](ORGANIZATION)"
        nlp (spacy.Language, optional): spaCy language model. If None, uses blank English model.
        style (str): displaCy style ("ent" or "dep")
        jupyter (bool): Whether to render for Jupyter notebook

    Returns:
        Rendered visualization (HTML string if not in Jupyter)
    """
    doc = annotated_text_to_spacy_doc(text, nlp)

    try:
        import spacy
        return spacy.displacy.render(doc, style=style, jupyter=jupyter)
    except ImportError:
        print("spaCy not installed. Please install with: pip install spacy")
        return None

In [8]:
def parse_annotated_text_to_dataframe(text):
    """
    Parses annotated text in format [Entity](LABEL) and converts it to a pandas DataFrame.

    Args:
        text (str): Text with annotations like "[Tom](PERSON) worked for [Microsoft](ORGANIZATION)"

    Returns:
        pandas.DataFrame: DataFrame with columns 'entity_text' and 'label'
    """
    pattern = r'\[([^\]]+)\]\(([^)]+)\)'
    entities_list = []

    for match in re.finditer(pattern, text):
        entity_text = match.group(1)
        entity_label = match.group(2)
        entities_list.append({"entity_text": entity_text, "label": entity_label})

    df = pd.DataFrame(entities_list)
    return df

## Input variables for the prompt

Here, we define:
- Two possible text input: A simple text, and a series of dictionaries containing a text tokenized into sentences as input data for our NER prompt.
- A set of LABELS to use.

Alternatively, we can run the process on a local file, which, however, we should tokenize to ensure accuracy in the result. See the code below.

In [9]:
# @title
# two options for the input: TEXT_DATA is a simple string, while INPUT_DATA contains the text already tokenized by sentences.

TEXT_DATA = "This painting depicts Monet's first wife, Camille, outside on a snowy day passing by the French doors of their home at Argenteuil. Her face is rendered in a radically bold Impressionist technique of mere daubs of paint quickly applied, just as the snow and trees are defined by broad, broken strokes of pure white and green."
INPUT_DATA = [{'text_original': "This painting depicts Monet's first wife, Camille, outside on a snowy day passing by the French doors of their home at Argenteuil. Her face is rendered in a radically bold Impressionist technique of mere daubs of paint quickly applied, just as the snow and trees are defined by broad, broken strokes of pure white and green.",
  'text_clean': "This painting depicts Monet's first wife, Camille, outside on a snowy day passing by the French doors of their home at Argenteuil. Her face is rendered in a radically bold Impressionist technique of mere daubs of paint quickly applied, just as the snow and trees are defined by broad, broken strokes of pure white and green.",
  'language': {'language': 'en', 'score': -868.9007034301758},
  'sentences': [{'id': 0,
    'start': 0,
    'end': 130,
    'text': "This painting depicts Monet's first wife, Camille, outside on a snowy day passing by the French doors of their home at Argenteuil."},
   {'id': 1,
    'start': 131,
    'end': 324,
    'text': 'Her face is rendered in a radically bold Impressionist technique of mere daubs of paint quickly applied, just as the snow and trees are defined by broad, broken strokes of pure white and green.'}],
  'tokens': [{'id': 0,
    'text': 'This',
    'start': 0,
    'end': 4,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 1,
    'text': 'painting',
    'start': 5,
    'end': 13,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 2,
    'text': 'depicts',
    'start': 14,
    'end': 21,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 3,
    'text': 'Monet',
    'start': 22,
    'end': 27,
    'ws': False,
    'is_punct': False,
    'sent_id': 0},
   {'id': 4,
    'text': "'s",
    'start': 27,
    'end': 29,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 5,
    'text': 'first',
    'start': 30,
    'end': 35,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 6,
    'text': 'wife',
    'start': 36,
    'end': 40,
    'ws': False,
    'is_punct': False,
    'sent_id': 0},
   {'id': 7,
    'text': ',',
    'start': 40,
    'end': 41,
    'ws': True,
    'is_punct': True,
    'sent_id': 0},
   {'id': 8,
    'text': 'Camille',
    'start': 42,
    'end': 49,
    'ws': False,
    'is_punct': False,
    'sent_id': 0},
   {'id': 9,
    'text': ',',
    'start': 49,
    'end': 50,
    'ws': True,
    'is_punct': True,
    'sent_id': 0},
   {'id': 10,
    'text': 'outside',
    'start': 51,
    'end': 58,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 11,
    'text': 'on',
    'start': 59,
    'end': 61,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 12,
    'text': 'a',
    'start': 62,
    'end': 63,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 13,
    'text': 'snowy',
    'start': 64,
    'end': 69,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 14,
    'text': 'day',
    'start': 70,
    'end': 73,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 15,
    'text': 'passing',
    'start': 74,
    'end': 81,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 16,
    'text': 'by',
    'start': 82,
    'end': 84,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 17,
    'text': 'the',
    'start': 85,
    'end': 88,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 18,
    'text': 'French',
    'start': 89,
    'end': 95,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 19,
    'text': 'doors',
    'start': 96,
    'end': 101,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 20,
    'text': 'of',
    'start': 102,
    'end': 104,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 21,
    'text': 'their',
    'start': 105,
    'end': 110,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 22,
    'text': 'home',
    'start': 111,
    'end': 115,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 23,
    'text': 'at',
    'start': 116,
    'end': 118,
    'ws': True,
    'is_punct': False,
    'sent_id': 0},
   {'id': 24,
    'text': 'Argenteuil',
    'start': 119,
    'end': 129,
    'ws': False,
    'is_punct': False,
    'sent_id': 0},
   {'id': 25,
    'text': '.',
    'start': 129,
    'end': 130,
    'ws': True,
    'is_punct': True,
    'sent_id': 0},
   {'id': 26,
    'text': 'Her',
    'start': 131,
    'end': 134,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 27,
    'text': 'face',
    'start': 135,
    'end': 139,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 28,
    'text': 'is',
    'start': 140,
    'end': 142,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 29,
    'text': 'rendered',
    'start': 143,
    'end': 151,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 30,
    'text': 'in',
    'start': 152,
    'end': 154,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 31,
    'text': 'a',
    'start': 155,
    'end': 156,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 32,
    'text': 'radically',
    'start': 157,
    'end': 166,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 33,
    'text': 'bold',
    'start': 167,
    'end': 171,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 34,
    'text': 'Impressionist',
    'start': 172,
    'end': 185,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 35,
    'text': 'technique',
    'start': 186,
    'end': 195,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 36,
    'text': 'of',
    'start': 196,
    'end': 198,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 37,
    'text': 'mere',
    'start': 199,
    'end': 203,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 38,
    'text': 'daubs',
    'start': 204,
    'end': 209,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 39,
    'text': 'of',
    'start': 210,
    'end': 212,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 40,
    'text': 'paint',
    'start': 213,
    'end': 218,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 41,
    'text': 'quickly',
    'start': 219,
    'end': 226,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 42,
    'text': 'applied',
    'start': 227,
    'end': 234,
    'ws': False,
    'is_punct': False,
    'sent_id': 1},
   {'id': 43,
    'text': ',',
    'start': 234,
    'end': 235,
    'ws': True,
    'is_punct': True,
    'sent_id': 1},
   {'id': 44,
    'text': 'just',
    'start': 236,
    'end': 240,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 45,
    'text': 'as',
    'start': 241,
    'end': 243,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 46,
    'text': 'the',
    'start': 244,
    'end': 247,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 47,
    'text': 'snow',
    'start': 248,
    'end': 252,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 48,
    'text': 'and',
    'start': 253,
    'end': 256,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 49,
    'text': 'trees',
    'start': 257,
    'end': 262,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 50,
    'text': 'are',
    'start': 263,
    'end': 266,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 51,
    'text': 'defined',
    'start': 267,
    'end': 274,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 52,
    'text': 'by',
    'start': 275,
    'end': 277,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 53,
    'text': 'broad',
    'start': 278,
    'end': 283,
    'ws': False,
    'is_punct': False,
    'sent_id': 1},
   {'id': 54,
    'text': ',',
    'start': 283,
    'end': 284,
    'ws': True,
    'is_punct': True,
    'sent_id': 1},
   {'id': 55,
    'text': 'broken',
    'start': 285,
    'end': 291,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 56,
    'text': 'strokes',
    'start': 292,
    'end': 299,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 57,
    'text': 'of',
    'start': 300,
    'end': 302,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 58,
    'text': 'pure',
    'start': 303,
    'end': 307,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 59,
    'text': 'white',
    'start': 308,
    'end': 313,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 60,
    'text': 'and',
    'start': 314,
    'end': 317,
    'ws': True,
    'is_punct': False,
    'sent_id': 1},
   {'id': 61,
    'text': 'green',
    'start': 318,
    'end': 323,
    'ws': False,
    'is_punct': False,
    'sent_id': 1},
   {'id': 62,
    'text': '.',
    'start': 323,
    'end': 324,
    'ws': False,
    'is_punct': True,
    'sent_id': 1}],
  'meta': {'source': 'CMA',
   'id': 135382,
   'char_count': 324,
   'token_count': 63,
   'sentence_count': 2}}]

In [10]:
# define the variables to use in the prompt
LABELS = ["PERSON", "LOCATION", "ORGANIZATION"]

In [11]:
# define the text input to use in the prompt
TEXT = INPUT_DATA[0]["text_clean"] # the TEXT variable for our input file comes from the cleaned version of our text, already available in our input data.
print(TEXT)

This painting depicts Monet's first wife, Camille, outside on a snowy day passing by the French doors of their home at Argenteuil. Her face is rendered in a radically bold Impressionist technique of mere daubs of paint quickly applied, just as the snow and trees are defined by broad, broken strokes of pure white and green.


## Crafting the Prompt

In [12]:
prompt = f"""
Convert the following text into a structured markdown format, where you annotate the entities in the text in the following format: [Tom](PERSON) went to [New York](PLACE).

Look for the following entities types:
{LABELS}

Do this for the following text:
{TEXT}

Only return the markdown output, nothing else.
"""

In [13]:
print(prompt) # test the prompt to make sure it picked the correct variables


Convert the following text into a structured markdown format, where you annotate the entities in the text in the following format: [Tom](PERSON) went to [New York](PLACE).

Look for the following entities types:
['PERSON', 'LOCATION', 'ORGANIZATION']

Do this for the following text:
This painting depicts Monet's first wife, Camille, outside on a snowy day passing by the French doors of their home at Argenteuil. Her face is rendered in a radically bold Impressionist technique of mere daubs of paint quickly applied, just as the snow and trees are defined by broad, broken strokes of pure white and green.

Only return the markdown output, nothing else.



## Calling Gemini

In [14]:
# run NER on our input text using the prompt we defined above.

response = model.generate_content(prompt)
output_text = response.text
print(output_text)

This painting depicts [Monet](PERSON)'s first wife, [Camille](PERSON), outside on a snowy day passing by the French doors of their home at [Argenteuil](LOCATION). Her face is rendered in a radically bold Impressionist technique of mere daubs of paint quickly applied, just as the snow and trees are defined by broad, broken strokes of pure white and green.


In [15]:
# convert the output into markdown format that can be processed through displaCy.
markdown_output = output_text
print(markdown_output)

This painting depicts [Monet](PERSON)'s first wife, [Camille](PERSON), outside on a snowy day passing by the French doors of their home at [Argenteuil](LOCATION). Her face is rendered in a radically bold Impressionist technique of mere daubs of paint quickly applied, just as the snow and trees are defined by broad, broken strokes of pure white and green.


## Visualizing the Results

In [16]:
visualize_annotated_text(markdown_output)

In [17]:
doc = annotated_text_to_spacy_doc(markdown_output)
print(doc.ents)

(Monet, Camille, Argenteuil)


In [18]:
entities = []
for ent in doc.ents:
    print(ent.text, ent.label_, ent.start_char, ent.end_char)
    entities.append({
        "text": ent.text,
        "label": ent.label_,
        "start_char": ent.start_char,
        "end_char": ent.end_char
    })


Monet PERSON 22 27
Camille PERSON 43 50
Argenteuil LOCATION 121 131


In [19]:
INPUT_DATA[0]["entities"] = entities
print(INPUT_DATA[0]["entities"])

[{'text': 'Monet', 'label': 'PERSON', 'start_char': 22, 'end_char': 27}, {'text': 'Camille', 'label': 'PERSON', 'start_char': 43, 'end_char': 50}, {'text': 'Argenteuil', 'label': 'LOCATION', 'start_char': 121, 'end_char': 131}]


In [20]:
parse_annotated_text_to_dataframe(markdown_output)

,entity_text,label
0,Monet,PERSON
1,Camille,PERSON
2,Argenteuil,LOCATION


# For large files: tokenization and NER on single sentences

Finally, we may want to run NER on a local file, perhaps a large one. To ensure accurate results, we will tokenize the file into sentences using using spaCy, and iterate our prompt.

### Preparing the text and defining the prompt

In [21]:
TEXT_FILE = "/content/thucydides_histories_sampleparagraph.txt"
with open(TEXT_FILE, "r", encoding="utf-8") as f:
    input_text = f.read()

LABELS = ["PERSON", "LOCATION", "ORGANIZATION"]
TEXT = input_text
print(TEXT)

§ 1.1 Thucydides, an Athenian, wrote the history of the war between the Peloponnesians and the Athenians, beginning at the moment that it broke out, and believing that it would be a great war, and more worthy of relation than any that had preceded it. This belief was not without its grounds. The preparations of both the combatants were in every department in the last state of perfection; and he could see the rest of the Hellenic race taking sides in the quarrel; those who delayed doing so at once having it in contemplation. 2 Indeed this was the greatest movement yet known in history, not only of the Hellenes, but of a large part of the barbarian world — I had almost said of mankind. 3 For though the events of remote antiquity, and even those that more immediately precede the war, could not from lapse of time be clearly ascertained, yet the evidences which an inquiry carried as far back as was practicable leads me to trust, all point to the conclusion that there was nothing on a great 

In [22]:
# use the spacy nlp() module to tokenize the file into sentences
nlp = spacy.load("en_core_web_sm")

doc = nlp(TEXT)
# extract the text of the tokenized sentences
sentences = [sent.text for sent in doc.sents]
print(sentences)

['§ 1.1 Thucydides, an Athenian, wrote the history of the war between the Peloponnesians and the Athenians, beginning at the moment that it broke out, and believing that it would be a great war, and more worthy of relation than any that had preceded it.', 'This belief was not without its grounds.', 'The preparations of both the combatants were in every department in the last state of perfection; and he could see the rest of the Hellenic race taking sides in the quarrel; those who delayed doing so at once having it in contemplation.', '2', 'Indeed this was the greatest movement yet known in history, not only of the Hellenes, but of a large part of the barbarian world — I had almost said of mankind.', '3', "For though the events of remote antiquity, and even those that more immediately precede the war, could not from lapse of time be clearly ascertained, yet the evidences which an inquiry carried as far back as was practicable leads me to trust, all point to the conclusion that there was

In [23]:
prompt_sentences = f"""
Convert the following text into a structured markdown format, where you annotate the entities in the text in the following format: [Tom](PERSON) went to [New York](PLACE).

Look for the following entities types:
{LABELS}

Only return the markdown output, nothing else.
"""

## Calling Gemini

Here we iterate the prompt over the sentences of the file, producing a response for each sentence.

In [24]:
# sentence-based NER processing function
def ner_on_file(sentences, prompt):
    ner_results = []
    for sentence in sentences:
        prompt = f"""{prompt_sentences}. Do this for the following text: {sentence}"""
        response = model.generate_content(prompt)
        ner_results.append({
            "response": response.text
        })

        print(f"Response: {response.text}\n")
    return ner_results

In [ ]:
ner_output = ner_on_file(sentences, prompt_sentences)

## Visualizing the Results

In [ ]:
# print the results as markdown annotations

markdown_output = ""
for item in ner_output:
    markdown_output += item["response"] + "\n"
print(markdown_output)

In [ ]:
# print the markdown output as a pandas dataframe

df = parse_annotated_text_to_dataframe(markdown_output)
display(df)

In [ ]:
# visualize the results with displacy

visualize_annotated_text(markdown_output)

## Saving the results

In [ ]:
# save the results to a csv file
df.to_csv('entities.csv', index=False)

# save the results as a markdown document
with open("ner_output.md", "w") as f:
    f.write(markdown_output)